# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import gradio as gr
import json


In [ ]:
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    print("OPENAI_API_KEY not set up yet")
else:
    print("OPENAI_API_KEY has already set up")


In [ ]:
def add(num1: int, num2: int) -> int:
    """
        Add two numbers
    """
    print(f"call tool add: {num1} + {num2}")
    return num1 + num2


def sub(num1: int, num2: int) -> int:
    """
        Substract two numbers
    """
    print(f"call tool sub: {num1} - {num2}")
    return num1 - num2


def multiple(num1: int, num2: int) -> int:
    """
        Multiple two numbers
    """
    print(f"call too multiple: {num1} * {num2}")
    return num1 * num2


def divide(num1: int, num2: int) -> int:
    """
        Divide two numbers
    """
    print(f"call tool divide: {num1} / {num2}")
    if num2 == 0:
        raise ZeroDivisionError(f"{num2} cannot be 0")
    return num1 / num2

In [ ]:
# construct schema
add_function = {
    "name": "add",
    "description": "a function to add two numbers",
    "parameters": {
        "type": "object",
        "properties": {
            "num1": {
                "type": "number",
                "description": "first number to add"
            },
            "num2": {
                "type": "number",
                "description": "second number to add"
            },
        },
        "required": ["num1", "num2"],
        "additionalProperties": False,
    },
}


sub_function = {
    "name": "sub",
    "description": "a function to substract two numbers",
    "parameters": {
        "type": "object",
        "properties": {
            "num1": {
                "type": "number",
                "description": "first number to substract"
            },
            "num2": {
                "type": "number",
                "description": "second number to substract"
            },
        },
        "required": ["num1", "num2"],
        "additionalProperties": False,
    },
}


mul_function = {
    "name": "mul",
    "description": "a function to multiple two numbers",
    "parameters": {
        "type": "object",
        "properties": {
            "num1": {
                "type": "number",
                "description": "first number to multiple"
            },
            "num2": {
                "type": "number",
                "description": "second number to multiple"
            },
        },
        "required": ["num1", "num2"],
        "additionalProperties": False,
    },
}


divide_function = {
    "name": "divide",
    "description": "a function to divide two numbers",
    "parameters": {
        "type": "object",
        "properties": {
            "num1": {
                "type": "number",
                "description": "first number to divide"
            },
            "num2": {
                "type": "number",
                "description": "second number to divide"
            },
        },
        "required": ["num1", "num2"],
        "additionalProperties": False,
    },
}

In [ ]:
tools = [
    {"type": "function", "function": add_function},
    {"type": "function", "function": sub_function},
    {"type": "function", "function": mul_function},
    {"type": "function", "function": divide_function},   
]

In [ ]:
system_message=""" 
You are a helpful assistant for some math calculations called calculatorAI.
Give short, courteous answer, no more thatn I sentences.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
openai = OpenAI()
MODEL = "gpt-4.1-mini"

In [ ]:
def handle_tool_calls(message):
    """
    handle tool calls for calculate
    """
    responses = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        num1 = arguments.get("num1")
        num2 = arguments.get("num2")
        function_name = tool_call.function.name
        res = ""
        if function_name == "add":
            res = add(num1, num2)

        if function_name == "sub":
            res = sub(num1, num2)

        if function_name == "mul":
            res = multiple(num1, num2)

        if function_name == "divide":
            res = divide(num1, num2)
        res = f"{res}"
        if res:
            responses.append({
                "role": "tool",
                "content": res,
                "tool_call_id": tool_call.id
            })
    return responses


In [ ]:
def chat(message, history):
    """
    chat callback for gradio
    """
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":

        message = response.choices[0].message
        response = handle_tool_calls(message)
        messages = [*messages, *[message], *response]
        response = openai.chat.completions.create(model=MODEL, messages=messages)


    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()